# Poker Hand Prediction — Feature-Engineered RF & ANN




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import time

## 1. Load data

In [ ]:
df = pd.read_csv("poker_dataset.csv")
print(df.shape)

(1000000, 7)


## 2. Encode target labels

In [ ]:
handresult = {
    'NOTHING': 0,
    'PAIR': 1,
    'TWO PAIR': 2,
    'THREE OF A KIND': 3,
    'STRAIGHT': 4,
    'FLUSH': 5,
    'FULL HOUSE': 6,
    'FOUR OF A KIND': 7,
    'STRAIGHT FLUSH': 8,
    'ROYAL FLUSH': 9
}

df['result1'] = df['result1'].map(handresult).astype('int8')
df['result2'] = df['result2'].map(handresult).astype('int8')
df['result3'] = df['result3'].map(handresult).astype('int8')

## 3. Parse cards
Parse the unicode-suit card strings into `(suit, rank)` pairs once, up front.

In [ ]:
RANK_MAP = {'A':14,'K':13,'Q':12,'J':11,'10':10,'9':9,'8':8,'7':7,'6':6,'5':5,'4':4,'3':3,'2':2}
SUIT_MAP = {'♠':0, '♥':1, '♦':2, '♣':3}

def parse_cards(card_string):
    """'♦Q ♣3 ♥3 ♣7 ♣6' -> flat [suit,rank,suit,rank,...] in order"""
    out = []
    for token in card_string.split(' '):
        out.append(SUIT_MAP[token[0]])
        out.append(RANK_MAP[token[1:]])
    return out

t0 = time.time()
df['hand_parsed']  = df['hand'].apply(parse_cards)
df['river_parsed'] = df['river'].apply(parse_cards)
print("parse time:", round(time.time()-t0, 1), "s")

parse time: 3.3 s


## 4. Feature engineering + transform

For every stage (Preflop/Flop/Turn) we compute, from the cards actually visible at that point:
- `max_rank_count`, `num_pairs`, `has_trips`, `has_quads` — pairing structure
- `max_suit_count` — flush potential
- `distinct_ranks`, `distinct_suits`
- `high_card`, `low_card`, `rank_sum`, `rank_span`, `rank_std`
- `straight_potential` — longest run of consecutive ranks (Ace counted both high and low)
- `is_pocket_pair`, `is_suited_hole`, `hole_gap`, `hole_high` — hole-card-specific signals


In [ ]:
FEAT_NAMES = ['max_rank_count','num_pairs','has_trips','has_quads','max_suit_count',
              'distinct_ranks','distinct_suits','high_card','low_card','rank_sum',
              'rank_span','rank_std','straight_potential','is_pocket_pair',
              'is_suited_hole','hole_gap','hole_high','num_cards_seen']

def longest_run(rs):
    if not rs: return 0
    rs = sorted(set(rs)); best = 1; cur = 1
    for i in range(1, len(rs)):
        if rs[i] == rs[i-1] + 1:
            cur += 1; best = max(best, cur)
        else:
            cur = 1
    return best

def transform_dataframe(hands_df):
    """Expands each hand into 3 rows (Preflop/Flop/Turn) with raw card slots + engineered features."""
    N = len(hands_df); M = N * 3
    cols_rank = [f'c{i}_rank' for i in range(1, 8)]
    cols_suit = [f'c{i}_suit' for i in range(1, 8)]
    arrs = {c: np.zeros(M, dtype=np.int8) for c in cols_rank + cols_suit}
    for f in FEAT_NAMES:
        arrs[f] = np.zeros(M, dtype=np.float32 if f == 'rank_std' else np.int8)
    stage_arr = np.zeros(M, dtype=np.int8)
    target_arr = np.zeros(M, dtype=np.int8)

    hp_all = hands_df['hand_parsed'].to_numpy()
    rp_all = hands_df['river_parsed'].to_numpy()
    r1 = hands_df['result1'].to_numpy(); r2 = hands_df['result2'].to_numpy(); r3 = hands_df['result3'].to_numpy()

    idx = 0
    for n in range(N):
        hp = hp_all[n]
        hole = [(hp[0], hp[1]), (hp[2], hp[3])]
        rp = rp_all[n]
        community = [(rp[k], rp[k+1]) for k in range(0, 10, 2)]
        flop = community[:3]; turn = community[:4]
        h_ranks = [hole[0][1], hole[1][1]]; h_suits = [hole[0][0], hole[1][0]]
        is_pocket_pair = int(h_ranks[0] == h_ranks[1])
        is_suited_hole = int(h_suits[0] == h_suits[1])
        hole_gap = abs(h_ranks[0] - h_ranks[1])
        hole_high = max(h_ranks)
        for cards, stage_code, target in [([], 0, r1[n]), (flop, 1, r2[n]), (turn, 2, r3[n])]:
            all_cards = hole + cards
            padded = all_cards + [(0, 0)] * (7 - len(all_cards))
            for i, (suit, rank) in enumerate(padded, start=1):
                arrs[f'c{i}_rank'][idx] = rank; arrs[f'c{i}_suit'][idx] = suit
            ranks = [rk for s, rk in all_cards]; suits = [s for s, rk in all_cards]
            rc = Counter(ranks); sc = Counter(suits)
            arrs['max_rank_count'][idx] = max(rc.values()) if rc else 0
            arrs['num_pairs'][idx] = sum(1 for c in rc.values() if c == 2)
            arrs['has_trips'][idx] = int(any(c >= 3 for c in rc.values()))
            arrs['has_quads'][idx] = int(any(c >= 4 for c in rc.values()))
            arrs['max_suit_count'][idx] = max(sc.values()) if sc else 0
            arrs['distinct_ranks'][idx] = len(rc); arrs['distinct_suits'][idx] = len(sc)
            arrs['high_card'][idx] = max(ranks) if ranks else 0
            arrs['low_card'][idx] = min(ranks) if ranks else 0
            arrs['rank_sum'][idx] = min(sum(ranks), 127)
            arrs['rank_span'][idx] = (max(ranks) - min(ranks)) if ranks else 0
            arrs['rank_std'][idx] = float(np.std(ranks)) if len(ranks) > 1 else 0.0
            run1 = longest_run(ranks); alt = [1 if rk == 14 else rk for rk in ranks]
            arrs['straight_potential'][idx] = max(run1, longest_run(alt))
            arrs['is_pocket_pair'][idx] = is_pocket_pair
            arrs['is_suited_hole'][idx] = is_suited_hole
            arrs['hole_gap'][idx] = hole_gap
            arrs['hole_high'][idx] = hole_high
            arrs['num_cards_seen'][idx] = len(all_cards)
            stage_arr[idx] = stage_code; target_arr[idx] = target
            idx += 1
    data = {**arrs}
    data['stage'] = stage_arr
    data['target'] = target_arr
    return pd.DataFrame(data)

t0 = time.time()
full_transformed = transform_dataframe(df)
print("transform time:", round(time.time()-t0, 1), "s")
print(full_transformed.shape)

transform time: 112.8 s
(3000000, 34)


## 5. Hand-level train/test split


In [ ]:
SAMPLE_HANDS = 100000   # sandbox-safe subsample (3.9GB RAM / 1 CPU). Set to None to use all 1,000,000 hands.
RANDOM_STATE = 42

from sklearn.model_selection import train_test_split

n_hands_total = len(full_transformed) // 3
print("total original hands:", n_hands_total)

hand_ids = np.arange(n_hands_total)
if SAMPLE_HANDS is not None:
    rng = np.random.RandomState(RANDOM_STATE)
    hand_ids = rng.choice(hand_ids, size=SAMPLE_HANDS, replace=False)

train_hand_ids, test_hand_ids = train_test_split(hand_ids, test_size=0.2, random_state=RANDOM_STATE)

def rows_for_hands(hids):
    idx = np.concatenate([np.array([3*h, 3*h+1, 3*h+2]) for h in hids])
    return full_transformed.iloc[idx]

train_df = rows_for_hands(train_hand_ids).reset_index(drop=True)
test_df  = rows_for_hands(test_hand_ids).reset_index(drop=True)
print("train_df:", train_df.shape, "test_df:", test_df.shape)

total original hands: 1000000
train_df: (240000, 34) test_df: (60000, 34)


## 6. One-hot encode suits + stage


In [ ]:
suit_cols = [f'c{i}_suit' for i in range(1, 8)]

train_oh = pd.get_dummies(train_df, columns=suit_cols + ['stage'], prefix=suit_cols + ['stage'])
test_oh  = pd.get_dummies(test_df,  columns=suit_cols + ['stage'], prefix=suit_cols + ['stage'])
test_oh  = test_oh.reindex(columns=train_oh.columns, fill_value=0)

X_train = train_oh.drop(columns=['target']); y_train_raw = train_oh['target']
X_test  = test_oh.drop(columns=['target']);  y_test_raw  = test_oh['target']
print("X_train:", X_train.shape, "X_test:", X_test.shape)

X_train: (240000, 53) X_test: (60000, 53)


## 7. Label-encode the target

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
n_classes = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)

Classes: [0 1 2 3 4 5 6 7 8 9]


## 8. Random Forest — hyperparameter search


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score

param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [15, 20, 25],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced', None],
}

rf = RandomForestClassifier(random_state=42, n_jobs=1)  # n_jobs=-1 on your machine if you have more cores
rf_search = RandomizedSearchCV(
    estimator=rf, param_distributions=param_dist, n_iter=6,
    scoring='accuracy', cv=2, verbose=2, random_state=42, n_jobs=1
)

t0 = time.time()
rf_search.fit(X_train, y_train)
print("search time:", round(time.time()-t0, 1), "s")
print("Best RF params:", rf_search.best_params_)

Fitting 2 folds for each of 6 candidates, totalling 12 fits
[CV] END class_weight=balanced, max_depth=25, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=  19.0s
[CV] END class_weight=balanced, max_depth=25, max_features=sqrt, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=  19.1s
[CV] END class_weight=balanced, max_depth=25, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  24.0s
[CV] END class_weight=balanced, max_depth=25, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  22.8s
[CV] END class_weight=balanced, max_depth=25, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  44.5s
[CV] END class_weight=balanced, max_depth=25, max_features=log2, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=  44.3s
[CV] END class_weight=balanced, max_depth=15, max_features=sqrt, min_samples_lea

## 9. Random Forest — evaluation

In [ ]:
best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)
print("RF Test Accuracy:", rf_acc)
print(classification_report(y_test, y_pred_rf, target_names=[str(c) for c in label_encoder.classes_]))

RF Test Accuracy: 0.6300166666666667
              precision    recall  f1-score   support

           0       0.57      0.94      0.71     20450
           1       0.68      0.53      0.60     26791
           2       0.79      0.35      0.48      7622
           3       0.80      0.44      0.57      2319
           4       0.97      0.31      0.47      1170
           5       0.81      0.26      0.39       818
           6       0.92      0.26      0.40       766
           7       1.00      0.30      0.47        56
           8       0.00      0.00      0.00         6
           9       0.00      0.00      0.00         2

    accuracy                           0.63     60000
   macro avg       0.65      0.34      0.41     60000
weighted avg       0.67      0.63      0.61     60000


Classes 8 and 9 (straight flush, royal flush) get ~0 recall — with only 100k sampled hands there are just a handful of examples of these in the test set (6 and 2 respectively), which matches their true rarity in poker (royal flush is ~1 in 650,000 hands). 

## 10. ANN — preprocessing

Scale features for the neural net (trees don't need this, but NNs do).

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.astype(np.float32))
X_test_scaled  = scaler.transform(X_test.astype(np.float32))

y_train_cat = to_categorical(y_train, num_classes=n_classes)
y_test_cat  = to_categorical(y_test, num_classes=n_classes)

## 11. ANN — model


In [ ]:
ann = models.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),

    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.15),

    layers.Dense(n_classes, activation='softmax')
])

ann.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=10, restore_best_weights=True, mode='max'
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy', factor=0.5, patience=4, min_lr=1e-5, mode='max'
)

## 12. ANN — train

In [ ]:
t0 = time.time()
history = ann.fit(
    X_train_scaled, y_train_cat,
    validation_split=0.1,
    epochs=60,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=2
)
print("fit time:", round(time.time()-t0, 1), "s")

Epoch 1/60
844/844 - 6s - 7ms/step - accuracy: 0.5812 - loss: 1.0501 - val_accuracy: 0.6338 - val_loss: 0.8313 - learning_rate: 0.0010
Epoch 2/60
844/844 - 5s - 6ms/step - accuracy: 0.6164 - loss: 0.8561 - val_accuracy: 0.6323 - val_loss: 0.8230 - learning_rate: 0.0010
Epoch 3/60
844/844 - 5s - 6ms/step - accuracy: 0.6219 - loss: 0.8421 - val_accuracy: 0.6340 - val_loss: 0.8215 - learning_rate: 0.0010
Epoch 4/60
844/844 - 5s - 6ms/step - accuracy: 0.6243 - loss: 0.8341 - val_accuracy: 0.6339 - val_loss: 0.8188 - learning_rate: 0.0010
Epoch 5/60
844/844 - 5s - 6ms/step - accuracy: 0.6256 - loss: 0.8292 - val_accuracy: 0.6345 - val_loss: 0.8162 - learning_rate: 0.0010
Epoch 6/60
844/844 - 5s - 6ms/step - accuracy: 0.6267 - loss: 0.8262 - val_accuracy: 0.6348 - val_loss: 0.8160 - learning_rate: 0.0010
Epoch 7/60
844/844 - 5s - 6ms/step - accuracy: 0.6276 - loss: 0.8242 - val_accuracy: 0.6336 - val_loss: 0.8155 - learning_rate: 0.0010
Epoch 8/60
844/844 - 5s - 6ms/step - accuracy: 0.6279 -

## 13. ANN — evaluate

In [ ]:
ann_test_loss, ann_test_acc = ann.evaluate(X_test_scaled, y_test_cat, verbose=0)
print("ANN Test Accuracy:", ann_test_acc)

ANN Test Accuracy: 0.6334666609764099


## 14. Summary



Both models land in the same ballpark once the ANN has the same engineered features RF benefits from — which makes sense, since both are now learning from the same informative signals rather than the ANN having to rediscover poker combinatorics from raw cards alone. On your full 1,000,000-hand dataset with a fuller RF search, expect both numbers to move higher, and rare-hand recall (classes 8/9) to become meaningful instead of ~0.

## 15. Save models

In [ ]:
import joblib

joblib.dump(best_rf, 'random_forest_model.joblib')
joblib.dump(label_encoder, 'label_encoder.joblib')

ann.save('ann_model.keras')
joblib.dump(scaler, 'scaler.joblib')

print("Saved: random_forest_model.joblib, label_encoder.joblib, ann_model.keras, scaler.joblib")

Saved: random_forest_model.joblib, label_encoder.joblib, ann_model.keras, scaler.joblib
